In [1]:
import pandas as pd

# Load raw dataset
file_path = r"C:\Users\Lenovo\Desktop\Mahak Project\Marketing Campaign Business Performance Analytics\data\raw\ppc_campaign_performance_data.xlsx"

df = pd.read_excel(file_path)

# Convert date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Recalculate business KPIs from raw values
df["Calculated_CTR"] = df["Clicks"].div(df["Impressions"]).fillna(0)

df["Calculated_CPC"] = df["Spend"].div(df["Clicks"]).fillna(0)

df["Calculated_Conversion_Rate"] = (
    df["Conversions"].div(df["Clicks"]).fillna(0)
)

df["Calculated_CPA"] = (
    df["Spend"].div(df["Conversions"]).fillna(0)
)

df["Calculated_ROAS"] = (
    df["Revenue"].div(df["Spend"]).fillna(0)
)

# Budget utilization
df["Budget_Utilization"] = (
    df["Spend"].div(df["Budget"]).replace([float("inf")], 0).fillna(0)
)

print("Cleaned dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

display(df.head())

Cleaned dataset shape: (1000, 25)
Date range: 2024-02-10 00:00:00 to 2025-02-09 00:00:00


,Campaign_ID,Budget,Clicks,CTR,CPC,Conversions,CPA,Conversion_Rate,Duration,Platform,...,Spend,ROAS,Date,Impressions,Calculated_CTR,Calculated_CPC,Calculated_Conversion_Rate,Calculated_CPA,Calculated_ROAS,Budget_Utilization
0,C3578,6390,401,0.0461,15.94,174,36.72,0.4339,20,Instagram,...,6453.9,4.31,2025-01-19,8698,0.046103,16.094514,0.433915,37.091379,4.313671,1.01
1,C6702,9870,1286,0.2860,7.67,821,12.02,0.6384,28,LinkedIn,...,10067.4,12.72,2025-01-22,4496,0.286032,7.828460,0.638414,12.262363,12.721855,1.02
2,C9725,7700,1684,0.2122,4.57,1060,7.26,0.6295,15,Instagram,...,7623.0,25.45,2024-07-23,7935,0.212224,4.526722,0.629454,7.191509,25.446675,0.99
3,C9472,8420,444,0.0961,18.96,308,27.34,0.6937,25,Google,...,8504.2,2.82,2024-04-20,4620,0.096104,19.153604,0.693694,27.611039,2.824957,1.01
4,C7601,8470,1912,0.3652,4.43,1428,5.93,0.7469,9,Google,...,8046.5,34.43,2024-08-07,5235,0.365234,4.208421,0.746862,5.634804,34.428882,0.95


In [2]:
print("Negative Spend:", (df["Spend"] < 0).sum())
print("Negative Revenue:", (df["Revenue"] < 0).sum())
print("Zero Impressions:", (df["Impressions"] == 0).sum())
print("Zero Clicks:", (df["Clicks"] == 0).sum())
print("Zero Conversions:", (df["Conversions"] == 0).sum())
print("Spend above Budget:", (df["Spend"] > df["Budget"]).sum())

Negative Spend: 0
Negative Revenue: 0
Zero Impressions: 0
Zero Clicks: 0
Zero Conversions: 0
Spend above Budget: 446


In [3]:
import os

output_path = r"C:\Users\Lenovo\Desktop\Mahak Project\Marketing Campaign Business Performance Analytics\data\processed\Marketing_Campaign_Cleaned.csv"

os.makedirs(
    r"C:\Users\Lenovo\Desktop\Mahak Project\Marketing Campaign Business Performance Analytics\data\processed",
    exist_ok=True
)

df.to_csv(output_path, index=False)

print("✅ Marketing_Campaign_Cleaned.csv created successfully!")

✅ Marketing_Campaign_Cleaned.csv created successfully!


In [4]:
# Campaigns that exceeded their budget
over_budget = df[df["Spend"] > df["Budget"]].copy()

print("Over-budget campaigns:", len(over_budget))
print(
    "Percentage of campaigns over budget:",
    round(len(over_budget) / len(df) * 100, 2),
    "%"
)

display(
    over_budget[
        [
            "Campaign_ID",
            "Budget",
            "Spend",
            "Budget_Utilization",
            "Conversions",
            "Revenue",
            "Calculated_CPA",
            "Calculated_ROAS"
        ]
    ].head(10)
)

Over-budget campaigns: 446
Percentage of campaigns over budget: 44.6 %


,Campaign_ID,Budget,Spend,Budget_Utilization,Conversions,Revenue,Calculated_CPA,Calculated_ROAS
0,C3578,6390,6453.9,1.01,174,27840,37.091379,4.313671
1,C6702,9870,10067.4,1.02,821,128076,12.262363,12.721855
3,C9472,8420,8504.2,1.01,308,24024,27.611039,2.824957
6,C8817,4420,4508.4,1.02,452,51528,9.974336,11.429332
8,C9333,2070,2111.4,1.02,192,15744,10.996875,7.456664
12,C4133,7460,7609.2,1.02,154,22792,49.410390,2.995321
13,C5129,9490,9774.7,1.03,1288,136528,7.589053,13.967487
14,C2514,2810,2950.5,1.05,1068,206124,2.762640,69.860702
17,C6840,3290,3388.7,1.03,115,17020,29.466957,5.022575
22,C8963,4550,4641.0,1.02,139,19460,33.388489,4.193062


In [5]:
df["Budget_Overrun"] = (df["Spend"] - df["Budget"]).clip(lower=0)

df["Budget_Status"] = df["Budget_Utilization"].apply(
    lambda x: "Over Budget" if x > 1 else "Within Budget"
)

In [6]:
display(
    df.groupby("Budget_Status").agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Total_Conversions=("Conversions", "sum")
    )
)

,Campaigns,Total_Budget,Total_Spend,Total_Revenue,Total_Conversions
Budget_Status,,,,,
Over Budget,446,2640020,2718699.0,27054240,229457
Within Budget,554,3321970,3237468.8,32832470,276215


In [26]:
platform_analysis = (
    df.groupby("Platform")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

platform_analysis["Conversion_Rate"] = (
    platform_analysis["Total_Conversions"]
    / platform_analysis["Total_Clicks"]
)

platform_analysis["CTR"] = (
    platform_analysis["Total_Clicks"]
    / platform_analysis["Total_Impressions"]
)

platform_analysis["CPC"] = (
    platform_analysis["Total_Spend"]
    / platform_analysis["Total_Clicks"]
)

platform_analysis["CPA"] = (
    platform_analysis["Total_Spend"]
    / platform_analysis["Total_Conversions"]
)

platform_analysis["ROAS"] = (
    platform_analysis["Total_Revenue"]
    / platform_analysis["Total_Spend"]
)

platform_analysis["Budget_Utilization"] = (
    platform_analysis["Total_Spend"]
    / platform_analysis["Total_Budget"]
)

platform_analysis = platform_analysis.sort_values(
    "ROAS",
    ascending=False
)

display(platform_analysis)

,Platform,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,Conversion_Rate,CTR,CPC,CPA,ROAS,Budget_Utilization
0,Facebook,198,1233780,1233654.8,1053727,209241,110760,13846011,0.529342,0.198572,5.895856,11.138090,11.223570,0.999899
1,Google,183,1095430,1095212.2,1048806,191658,99241,11418956,0.517803,0.182739,5.714409,11.035884,10.426250,0.999801
3,LinkedIn,224,1349970,1347518.5,1302909,236744,114614,13426760,0.484126,0.181704,5.691880,11.757015,9.964064,0.998184
2,Instagram,200,1162500,1160623.2,1078363,186670,91304,11063115,0.489120,0.173105,6.217513,12.711636,9.532047,0.998386
4,YouTube,195,1120310,1119159.1,1013372,188916,89753,10131868,0.475095,0.186423,5.924110,12.469322,9.053108,0.998973


In [7]:
content_analysis = (
    df.groupby("Content_Type")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

content_analysis["CTR"] = (
    content_analysis["Total_Clicks"]
    / content_analysis["Total_Impressions"]
)

content_analysis["Conversion_Rate"] = (
    content_analysis["Total_Conversions"]
    / content_analysis["Total_Clicks"]
)

content_analysis["CPC"] = (
    content_analysis["Total_Spend"]
    / content_analysis["Total_Clicks"]
)

content_analysis["CPA"] = (
    content_analysis["Total_Spend"]
    / content_analysis["Total_Conversions"]
)

content_analysis["ROAS"] = (
    content_analysis["Total_Revenue"]
    / content_analysis["Total_Spend"]
)

display(
    content_analysis.sort_values(
        "ROAS",
        ascending=False
    )
)

,Content_Type,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,CTR,Conversion_Rate,CPC,CPA,ROAS
3,Video,247,1433010,1434884.0,1370540,248769,129314,15512932,0.181512,0.519816,5.767937,11.096123,10.811280
0,Carousel,266,1570050,1567257.1,1448373,267354,138291,16852746,0.184589,0.517258,5.862105,11.333038,10.753019
1,Image,234,1444210,1446641.3,1307809,245270,120500,14051159,0.187543,0.491295,5.898158,12.005322,9.712953
2,Text,253,1514720,1507385.4,1370455,251836,117567,13469873,0.183761,0.466840,5.985583,12.821501,8.935918


In [8]:
age_analysis = (
    df.groupby("Target_Age")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

age_analysis["CTR"] = (
    age_analysis["Total_Clicks"] /
    age_analysis["Total_Impressions"]
)

age_analysis["Conversion_Rate"] = (
    age_analysis["Total_Conversions"] /
    age_analysis["Total_Clicks"]
)

age_analysis["CPC"] = (
    age_analysis["Total_Spend"] /
    age_analysis["Total_Clicks"]
)

age_analysis["CPA"] = (
    age_analysis["Total_Spend"] /
    age_analysis["Total_Conversions"]
)

age_analysis["ROAS"] = (
    age_analysis["Total_Revenue"] /
    age_analysis["Total_Spend"]
)

display(
    age_analysis.sort_values(
        "ROAS",
        ascending=False
    )
)

,Target_Age,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,CTR,Conversion_Rate,CPC,CPA,ROAS
4,55+,199,1198240,1198637.3,1137403,200360,99473,12623989,0.176156,0.496471,5.982418,12.049876,10.531951
3,45-54,210,1251270,1250341.4,1087870,215294,107901,12776045,0.197904,0.501180,5.807600,11.587857,10.218045
1,25-34,199,1146490,1146310.5,1114606,202842,98905,11696020,0.181985,0.487596,5.651248,11.590016,10.203187
2,35-44,182,1031270,1026998.8,1008881,181808,87794,9914860,0.180208,0.482894,5.648810,11.697824,9.654208
0,18-24,210,1334720,1333879.8,1148417,212925,111599,12875796,0.185407,0.524124,6.264552,11.952435,9.652891


In [9]:
gender_analysis = (
    df.groupby("Target_Gender")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

gender_analysis["CTR"] = (
    gender_analysis["Total_Clicks"] /
    gender_analysis["Total_Impressions"]
)

gender_analysis["Conversion_Rate"] = (
    gender_analysis["Total_Conversions"] /
    gender_analysis["Total_Clicks"]
)

gender_analysis["CPC"] = (
    gender_analysis["Total_Spend"] /
    gender_analysis["Total_Clicks"]
)

gender_analysis["CPA"] = (
    gender_analysis["Total_Spend"] /
    gender_analysis["Total_Conversions"]
)

gender_analysis["ROAS"] = (
    gender_analysis["Total_Revenue"] /
    gender_analysis["Total_Spend"]
)

display(gender_analysis.sort_values("ROAS", ascending=False))

,Target_Gender,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,CTR,Conversion_Rate,CPC,CPA,ROAS
2,Other,349,2094970,2091655.0,1913209,367538,183659,23064332,0.192106,0.499701,5.690990,11.388797,11.026834
1,Male,335,1975000,1972332.7,1789631,336302,170102,19315755,0.187917,0.505801,5.864766,11.595000,9.793355
0,Female,316,1892020,1892180.1,1794337,309389,151911,17506623,0.172425,0.491003,6.115861,12.455847,9.252091


In [5]:
region_analysis = (
    df.groupby("Region")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

region_analysis["CTR"] = (
    region_analysis["Total_Clicks"] /
    region_analysis["Total_Impressions"]
)

region_analysis["Conversion_Rate"] = (
    region_analysis["Total_Conversions"] /
    region_analysis["Total_Clicks"]
)

region_analysis["CPC"] = (
    region_analysis["Total_Spend"] /
    region_analysis["Total_Clicks"]
)

region_analysis["CPA"] = (
    region_analysis["Total_Spend"] /
    region_analysis["Total_Conversions"]
)

region_analysis["ROAS"] = (
    region_analysis["Total_Revenue"] /
    region_analysis["Total_Spend"]
)

region_analysis["Budget_Utilization"] = (
    region_analysis["Total_Spend"] /
    region_analysis["Total_Budget"]
)

display(region_analysis.sort_values("ROAS", ascending=False))

,Region,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,CTR,Conversion_Rate,CPC,CPA,ROAS,Budget_Utilization
4,South America,202,1150280,1147589.4,1107042,206093,103165,12494158,0.186165,0.500575,5.568308,11.123825,10.887307,0.997661
1,Asia,187,1180340,1174890.5,988965,193464,106551,12572794,0.195623,0.550754,6.072915,11.026555,10.701247,0.995383
2,Europe,203,1192990,1191666.3,1097096,204957,101181,12499743,0.186818,0.493669,5.814226,11.777570,10.489298,0.998890
3,North America,200,1208100,1211399.9,1130302,202039,102702,12586455,0.178748,0.508328,5.995872,11.795290,10.390008,1.002731
0,Africa,208,1230280,1230621.7,1173772,206676,92073,9733560,0.176078,0.445494,5.954352,13.365717,7.909466,1.000278


In [14]:
campaign_analysis = df[
    [
        "Campaign_ID",
        "Platform",
        "Content_Type",
        "Target_Age",
        "Target_Gender",
        "Region",
        "Spend",
        "Conversions",
        "Revenue",
        "Calculated_CPA",
        "Calculated_ROAS"
    ]
].copy()

print("Top 10 campaigns by ROAS")
display(
    campaign_analysis.sort_values(
        "Calculated_ROAS",
        ascending=False
    ).head(10)
)

print("Top 10 campaigns by Revenue")
display(
    campaign_analysis.sort_values(
        "Revenue",
        ascending=False
    ).head(10)
)

print("Top 10 campaigns by Conversions")
display(
    campaign_analysis.sort_values(
        "Conversions",
        ascending=False
    ).head(10)
)

Top 10 campaigns by ROAS


,Campaign_ID,Platform,Content_Type,Target_Age,Target_Gender,Region,Spend,Conversions,Revenue,Calculated_CPA,Calculated_ROAS
831,C3181,YouTube,Image,25-34,Other,Asia,2381.6,1372,223636,1.735860,93.901579
259,C1783,Facebook,Image,25-34,Other,Europe,1976.0,1041,173847,1.898175,87.979251
875,C6062,Facebook,Video,18-24,Other,Europe,2987.6,1257,248886,2.376770,83.306333
30,C2608,LinkedIn,Image,55+,Male,South America,2777.5,1395,223200,1.991039,80.360036
92,C5887,Instagram,Video,45-54,Other,Europe,2111.5,869,155551,2.429804,73.668482
14,C2514,Facebook,Video,55+,Male,North America,2950.5,1068,206124,2.762640,69.860702
178,C8825,LinkedIn,Text,45-54,Male,Asia,2565.0,1138,171838,2.253954,66.993372
500,C5539,Google,Image,55+,Other,South America,2904.6,1374,193734,2.113974,66.699029
837,C9923,YouTube,Carousel,35-44,Female,Asia,2280.0,1048,143576,2.175573,62.971930
212,C5999,LinkedIn,Carousel,25-34,Other,North America,2514.6,854,157136,2.944496,62.489462


Top 10 campaigns by Revenue


,Campaign_ID,Platform,Content_Type,Target_Age,Target_Gender,Region,Spend,Conversions,Revenue,Calculated_CPA,Calculated_ROAS
218,C1002,Facebook,Carousel,18-24,Female,Asia,10348.0,1612,320788,6.419355,31.000000
127,C1011,Google,Carousel,55+,Other,South America,9027.2,1692,309636,5.335225,34.300337
119,C4053,Google,Video,35-44,Female,Asia,8918.0,1662,309132,5.365824,34.663826
100,C2290,Instagram,Carousel,55+,Other,Europe,7292.2,1559,308682,4.677486,42.330435
108,C1153,Instagram,Image,55+,Other,North America,8775.6,1562,306152,5.618182,34.886731
605,C5439,Instagram,Text,55+,Other,South America,9221.3,1550,300700,5.949226,32.609285
479,C5956,LinkedIn,Video,45-54,Other,North America,5180.0,1467,281664,3.531016,54.375290
4,C7601,Google,Text,25-34,Other,Europe,8046.5,1428,277032,5.634804,34.428882
703,C3177,LinkedIn,Image,55+,Other,Africa,8817.3,1606,276232,5.490224,31.328411
99,C2033,YouTube,Carousel,35-44,Other,Europe,7435.8,1424,271984,5.221770,36.577638


Top 10 campaigns by Conversions


,Campaign_ID,Platform,Content_Type,Target_Age,Target_Gender,Region,Spend,Conversions,Revenue,Calculated_CPA,Calculated_ROAS
111,C4095,LinkedIn,Image,45-54,Other,Africa,4771.2,1894,140156,2.519113,29.375419
246,C1725,Google,Text,18-24,Female,Asia,6880.5,1755,198315,3.920513,28.822760
912,C3443,Facebook,Video,45-54,Male,South America,9114.0,1752,129648,5.202055,14.225148
247,C3371,LinkedIn,Video,25-34,Other,North America,5740.8,1727,127798,3.324146,22.261357
322,C3107,Google,Carousel,18-24,Other,Africa,4569.6,1705,121055,2.680117,26.491378
127,C1011,Google,Carousel,55+,Other,South America,9027.2,1692,309636,5.335225,34.300337
723,C5704,Facebook,Carousel,25-34,Other,South America,4097.6,1690,253500,2.424615,61.865482
925,C9316,YouTube,Video,35-44,Female,Africa,4139.2,1675,108875,2.471164,26.303392
119,C4053,Google,Video,35-44,Female,Asia,8918.0,1662,309132,5.365824,34.663826
110,C4826,LinkedIn,Image,18-24,Female,Africa,9506.0,1641,206766,5.792809,21.751105


In [15]:
df["Budget_Status"] = pd.cut(
    df["Budget_Utilization"],
    bins=[-float("inf"), 0.95, 1.05, float("inf")],
    labels=["Under Budget", "Within Budget", "Over Budget"]
)

budget_analysis = (
    df.groupby("Budget_Status", observed=False)
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Total_Conversions=("Conversions", "sum")
    )
    .reset_index()
)

budget_analysis["CPA"] = (
    budget_analysis["Total_Spend"] /
    budget_analysis["Total_Conversions"]
)

budget_analysis["ROAS"] = (
    budget_analysis["Total_Revenue"] /
    budget_analysis["Total_Spend"]
)

display(budget_analysis)

,Budget_Status,Campaigns,Total_Budget,Total_Spend,Total_Revenue,Total_Conversions,CPA,ROAS
0,Under Budget,88,518050,492147.5,5109026,39387,12.495176,10.381087
1,Within Budget,912,5443940,5464020.3,54777684,466285,11.718199,10.025161
2,Over Budget,0,0,0.0,0,0,NaN,NaN


In [9]:
df["Month"] = df["Date"].dt.to_period("M").astype(str)

monthly_analysis = (
    df.groupby("Month")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

monthly_analysis["CTR"] = (
    monthly_analysis["Total_Clicks"] /
    monthly_analysis["Total_Impressions"]
)

monthly_analysis["Conversion_Rate"] = (
    monthly_analysis["Total_Conversions"] /
    monthly_analysis["Total_Clicks"]
)

monthly_analysis["CPC"] = (
    monthly_analysis["Total_Spend"] /
    monthly_analysis["Total_Clicks"]
)

monthly_analysis["CPA"] = (
    monthly_analysis["Total_Spend"] /
    monthly_analysis["Total_Conversions"]
)

monthly_analysis["ROAS"] = (
    monthly_analysis["Total_Revenue"] /
    monthly_analysis["Total_Spend"]
)

display(monthly_analysis)

,Month,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,CTR,Conversion_Rate,CPC,CPA,ROAS
0,2024-02,61,319380,319010.5,311261,58018,28443,3321877,0.186397,0.490244,5.498475,11.215782,10.413065
1,2024-03,87,521890,524689.3,499315,81129,40951,4927968,0.162481,0.504764,6.467346,12.812613,9.392164
2,2024-04,95,562600,563206.2,475986,100933,48081,6413032,0.212050,0.476366,5.580001,11.713696,11.386650
3,2024-05,84,540050,535933.1,459894,91331,49207,5769677,0.198591,0.538777,5.868031,10.891400,10.765666
4,2024-06,90,531790,531608.6,482623,85590,41201,4904379,0.177343,0.481376,6.211106,12.902808,9.225545
5,2024-07,87,519210,517916.0,439094,86963,44327,5116647,0.198051,0.509723,5.955590,11.683985,9.879299
6,2024-08,82,525380,523225.7,424083,83500,41453,5009860,0.196895,0.496443,6.266176,12.622143,9.574950
7,2024-09,75,450960,449067.5,406671,87993,40253,4747832,0.216374,0.457457,5.103446,11.156125,10.572647
8,2024-10,91,527780,528309.7,551379,88427,46476,4988722,0.160374,0.525586,5.974529,11.367366,9.442798
9,2024-11,64,388990,387299.2,376950,65303,37469,4478878,0.173240,0.573771,5.930803,10.336524,11.564387


In [10]:
platform_content_analysis = (
    df.groupby(["Platform", "Content_Type"])
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Spend=("Spend", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum")
    )
    .reset_index()
)

platform_content_analysis["CTR"] = (
    platform_content_analysis["Total_Clicks"]
    / platform_content_analysis["Total_Impressions"]
)

platform_content_analysis["Conversion_Rate"] = (
    platform_content_analysis["Total_Conversions"]
    / platform_content_analysis["Total_Clicks"]
)

platform_content_analysis["CPA"] = (
    platform_content_analysis["Total_Spend"]
    / platform_content_analysis["Total_Conversions"]
)

platform_content_analysis["ROAS"] = (
    platform_content_analysis["Total_Revenue"]
    / platform_content_analysis["Total_Spend"]
)

display(
    platform_content_analysis.sort_values(
        "ROAS",
        ascending=False
    )
)

,Platform,Content_Type,Campaigns,Total_Spend,Total_Revenue,Total_Conversions,Total_Impressions,Total_Clicks,CTR,Conversion_Rate,CPA,ROAS
3,Facebook,Video,47,279641.3,3708285,27482,260212,51350,0.197339,0.535190,10.175435,13.260863
4,Google,Carousel,51,301271.9,3533754,30540,274942,57602,0.209506,0.530190,9.864830,11.729451
0,Facebook,Carousel,49,318196.7,3688211,28898,264023,53313,0.201926,0.542044,11.011028,11.590978
15,LinkedIn,Video,48,280451.5,3164737,26154,263895,44139,0.167260,0.592537,10.723083,11.284436
16,YouTube,Carousel,50,276428.0,3069464,23486,237429,47731,0.201033,0.492049,11.769905,11.104027
13,LinkedIn,Image,67,401761.4,4304260,34575,414058,75477,0.182286,0.458087,11.619997,10.713473
5,Google,Image,34,206463.1,2210621,19681,210876,33938,0.160938,0.579910,10.490478,10.707100
11,Instagram,Video,54,299414.1,3077844,25210,305257,56271,0.184340,0.448011,11.876799,10.279556
7,Google,Video,44,260023.2,2672886,22137,257662,43575,0.169117,0.508021,11.746090,10.279414
1,Facebook,Image,44,292881.5,2991080,24850,220169,47928,0.217687,0.518486,11.785976,10.212595


In [16]:
audience_analysis = (
    df.groupby(
        ["Target_Age", "Target_Gender", "Region"]
    )
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Spend=("Spend", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum")
    )
    .reset_index()
)

audience_analysis["CTR"] = (
    audience_analysis["Total_Clicks"] /
    audience_analysis["Total_Impressions"]
)

audience_analysis["Conversion_Rate"] = (
    audience_analysis["Total_Conversions"] /
    audience_analysis["Total_Clicks"]
)

audience_analysis["CPA"] = (
    audience_analysis["Total_Spend"] /
    audience_analysis["Total_Conversions"]
)

audience_analysis["ROAS"] = (
    audience_analysis["Total_Revenue"] /
    audience_analysis["Total_Spend"]
)

display(
    audience_analysis.sort_values(
        "ROAS",
        ascending=False
    ).head(15)
)

,Target_Age,Target_Gender,Region,Campaigns,Total_Spend,Total_Revenue,Total_Conversions,Total_Impressions,Total_Clicks,CTR,Conversion_Rate,CPA,ROAS
74,55+,Other,South America,15,92280.3,1758571,10491,94240,18290,0.194079,0.573592,8.796140,19.056841
26,25-34,Other,Asia,14,81282.3,1515770,11922,66966,17541,0.261939,0.679665,6.817841,18.648217
1,18-24,Female,Asia,13,94532.1,1632823,12629,81257,17627,0.216929,0.716458,7.485320,17.272683
58,45-54,Other,North America,11,62593.5,1034192,7273,61760,13069,0.211609,0.556508,8.606284,16.522355
42,35-44,Other,Europe,12,61464.2,1005158,6726,74360,9647,0.129734,0.697212,9.138299,16.353552
54,45-54,Male,South America,22,130788.4,1959286,17207,112846,25614,0.226982,0.671781,7.600883,14.980579
63,55+,Female,North America,13,67634.0,974713,6348,72413,14580,0.201345,0.435391,10.654379,14.411583
38,35-44,Male,North America,13,61516.3,883062,8139,65852,13660,0.207435,0.595827,7.558214,14.354927
73,55+,Other,North America,13,66315.1,915594,6295,82654,10615,0.128427,0.593029,10.534567,13.806720
29,25-34,Other,South America,14,78045.9,1067948,7856,103288,16097,0.155846,0.488041,9.934560,13.683589


In [18]:
# Top 10 Audience Combinations by ROAS

top_10_audiences = (
    audience_analysis
    .sort_values("ROAS", ascending=False)
    .head(10)
    .copy()
)

# Create a readable combined audience label
top_10_audiences["Audience"] = (
    top_10_audiences["Target_Age"].astype(str)
    + " | "
    + top_10_audiences["Target_Gender"].astype(str)
    + " | "
    + top_10_audiences["Region"].astype(str)
)

# Keep only the columns needed for the chart
top_10_chart = top_10_audiences[
    ["Audience", "ROAS"]
].reset_index(drop=True)

display(top_10_chart)

,Audience,ROAS
0,55+ | Other | South America,19.056841
1,25-34 | Other | Asia,18.648217
2,18-24 | Female | Asia,17.272683
3,45-54 | Other | North America,16.522355
4,35-44 | Other | Europe,16.353552
5,45-54 | Male | South America,14.980579
6,55+ | Female | North America,14.411583
7,35-44 | Male | North America,14.354927
8,55+ | Other | North America,13.806720
9,25-34 | Other | South America,13.683589


In [13]:
platform_analysis = (
    df.groupby("Platform")
    .agg(
        Campaigns=("Campaign_ID", "count"),
        Total_Budget=("Budget", "sum"),
        Total_Spend=("Spend", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Clicks=("Clicks", "sum"),
        Total_Conversions=("Conversions", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

platform_analysis["Conversion_Rate"] = (
    platform_analysis["Total_Conversions"]
    / platform_analysis["Total_Clicks"]
)

platform_analysis["CTR"] = (
    platform_analysis["Total_Clicks"]
    / platform_analysis["Total_Impressions"]
)

platform_analysis["CPC"] = (
    platform_analysis["Total_Spend"]
    / platform_analysis["Total_Clicks"]
)

platform_analysis["CPA"] = (
    platform_analysis["Total_Spend"]
    / platform_analysis["Total_Conversions"]
)

platform_analysis["ROAS"] = (
    platform_analysis["Total_Revenue"]
    / platform_analysis["Total_Spend"]
)

display(platform_analysis)

,Platform,Campaigns,Total_Budget,Total_Spend,Total_Impressions,Total_Clicks,Total_Conversions,Total_Revenue,Conversion_Rate,CTR,CPC,CPA,ROAS
0,Facebook,198,1233780,1233654.8,1053727,209241,110760,13846011,0.529342,0.198572,5.895856,11.138090,11.223570
1,Google,183,1095430,1095212.2,1048806,191658,99241,11418956,0.517803,0.182739,5.714409,11.035884,10.426250
2,Instagram,200,1162500,1160623.2,1078363,186670,91304,11063115,0.489120,0.173105,6.217513,12.711636,9.532047
3,LinkedIn,224,1349970,1347518.5,1302909,236744,114614,13426760,0.484126,0.181704,5.691880,11.757015,9.964064
4,YouTube,195,1120310,1119159.1,1013372,188916,89753,10131868,0.475095,0.186423,5.924110,12.469322,9.053108


In [14]:
# ============================================
# Final Business Recommendations Summary
# ============================================

# 1. Best platform
best_platform = platform_analysis.loc[
    platform_analysis["ROAS"].idxmax()
]

# 2. Best platform + content combination
best_platform_content = platform_content_analysis.loc[
    platform_content_analysis["ROAS"].idxmax()
]

# 3. Best audience
best_audience = audience_analysis.loc[
    audience_analysis["ROAS"].idxmax()
]

# 4. Budget efficiency
best_budget_status = budget_analysis.loc[
    budget_analysis["ROAS"].idxmax()
]

# 5. Best month by ROAS
best_month = monthly_analysis.loc[
    monthly_analysis["ROAS"].idxmax()
]

# ============================================
# Display Findings
# ============================================

print("🎯 BUSINESS RECOMMENDATIONS")
print("=" * 60)

print("\n1. Platform Recommendation")
print(
    f"Prioritize {best_platform['Platform']} "
    f"with ROAS of {best_platform['ROAS']:.2f} "
    f"and conversion rate of {best_platform['Conversion_Rate']:.2%}."
)

print("\n2. Content Recommendation")
print(
    f"Prioritize {best_platform_content['Platform']} + "
    f"{best_platform_content['Content_Type']} "
    f"with ROAS of {best_platform_content['ROAS']:.2f}."
)

print("\n3. Audience Recommendation")
print(
    f"Focus on {best_audience['Target_Age']} + "
    f"{best_audience['Target_Gender']} + "
    f"{best_audience['Region']} "
    f"with ROAS of {best_audience['ROAS']:.2f}, "
    f"conversion rate of {best_audience['Conversion_Rate']:.2%}, "
    f"and CPA of {best_audience['CPA']:.2f}."
)

print("\n4. Budget Recommendation")
print(
    f"The strongest budget group by ROAS was "
    f"{best_budget_status['Budget_Status']} "
    f"with ROAS of {best_budget_status['ROAS']:.2f}."
)

print("\n5. Timing Recommendation")
print(
    f"{best_month['Month']} was the strongest month by ROAS "
    f"with ROAS of {best_month['ROAS']:.2f}, "
    f"revenue of {best_month['Total_Revenue']:,.0f}, "
    f"and CPA of {best_month['CPA']:.2f}."
)

🎯 BUSINESS RECOMMENDATIONS

1. Platform Recommendation
Prioritize Facebook with ROAS of 11.22 and conversion rate of 52.93%.

2. Content Recommendation
Prioritize Facebook + Video with ROAS of 13.26.

3. Audience Recommendation
Focus on 55+ + Other + South America with ROAS of 19.06, conversion rate of 57.36%, and CPA of 8.80.

4. Budget Recommendation
The strongest budget group by ROAS was Under Budget with ROAS of 10.38.

5. Timing Recommendation
2024-11 was the strongest month by ROAS with ROAS of 11.56, revenue of 4,478,878, and CPA of 10.34.


In [15]:
recommendations = pd.DataFrame({
    "Area": [
        "Platform",
        "Content",
        "Audience",
        "Budget",
        "Timing"
    ],
    "Recommendation": [
        f"Prioritize {best_platform['Platform']} for stronger marketing efficiency.",
        
        f"Prioritize {best_platform_content['Platform']} + "
        f"{best_platform_content['Content_Type']} campaigns.",
        
        f"Focus on {best_audience['Target_Age']} + "
        f"{best_audience['Target_Gender']} + "
        f"{best_audience['Region']} audiences.",
        
        f"Monitor {best_budget_status['Budget_Status'].lower()} campaigns "
        f"and reallocate spend based on ROAS and CPA.",
        
        f"Use {best_month['Month']} performance as a benchmark "
        f"for efficient campaign timing."
    ]
})

display(recommendations)

,Area,Recommendation
0,Platform,Prioritize Facebook for stronger marketing eff...
1,Content,Prioritize Facebook + Video campaigns.
2,Audience,Focus on 55+ + Other + South America audiences.
3,Budget,Monitor under budget campaigns and reallocate ...
4,Timing,Use 2024-11 performance as a benchmark for eff...


In [16]:
# ============================================
# Final KPI Selection
# ============================================

kpis = [
    "Total Campaigns",
    "Total Budget",
    "Total Spend",
    "Total Revenue",
    "Total Conversions",
    "Conversion Rate",
    "CPA",
    "ROAS"
]

kpi_table = pd.DataFrame({
    "KPI": kpis,
    "Purpose": [
        "Total number of marketing campaigns",
        "Total allocated marketing budget",
        "Total marketing spend",
        "Total revenue generated",
        "Total campaign conversions",
        "Percentage of clicks that converted",
        "Cost to acquire one conversion",
        "Revenue generated per unit of marketing spend"
    ]
})

display(kpi_table)

,KPI,Purpose
0,Total Campaigns,Total number of marketing campaigns
1,Total Budget,Total allocated marketing budget
2,Total Spend,Total marketing spend
3,Total Revenue,Total revenue generated
4,Total Conversions,Total campaign conversions
5,Conversion Rate,Percentage of clicks that converted
6,CPA,Cost to acquire one conversion
7,ROAS,Revenue generated per unit of marketing spend


In [17]:
# ============================================
# Overall KPI Values
# ============================================

total_campaigns = df["Campaign_ID"].nunique()
total_budget = df["Budget"].sum()
total_spend = df["Spend"].sum()
total_revenue = df["Revenue"].sum()
total_conversions = df["Conversions"].sum()
total_clicks = df["Clicks"].sum()
total_impressions = df["Impressions"].sum()

overall_ctr = total_clicks / total_impressions
overall_conversion_rate = total_conversions / total_clicks
overall_cpa = total_spend / total_conversions
overall_roas = total_revenue / total_spend

overall_kpis = pd.DataFrame({
    "KPI": [
        "Total Campaigns",
        "Total Budget",
        "Total Spend",
        "Total Revenue",
        "Total Conversions",
        "CTR",
        "Conversion Rate",
        "CPA",
        "ROAS"
    ],
    "Value": [
        total_campaigns,
        total_budget,
        total_spend,
        total_revenue,
        total_conversions,
        overall_ctr,
        overall_conversion_rate,
        overall_cpa,
        overall_roas
    ]
})

display(overall_kpis)

,KPI,Value
0,Total Campaigns,9.540000e+02
1,Total Budget,5.961990e+06
2,Total Spend,5.956168e+06
3,Total Revenue,5.988671e+07
4,Total Conversions,5.056720e+05
5,CTR,1.843181e-01
6,Conversion Rate,4.990698e-01
7,CPA,1.177872e+01
8,ROAS,1.005457e+01


In [19]:
# ============================================
# Campaign ID Duplicate Analysis
# ============================================

campaign_id_counts = (
    df["Campaign_ID"]
    .value_counts()
    .reset_index()
)

campaign_id_counts.columns = [
    "Campaign_ID",
    "Row_Count"
]

print("Total rows:", len(df))
print("Unique Campaign IDs:", df["Campaign_ID"].nunique())
print(
    "Campaign IDs appearing more than once:",
    (campaign_id_counts["Row_Count"] > 1).sum()
)

display(
    campaign_id_counts[
        campaign_id_counts["Row_Count"] > 1
    ].head(20)
)

Total rows: 1000
Unique Campaign IDs: 954
Campaign IDs appearing more than once: 45


,Campaign_ID,Row_Count
0,C5910,3
1,C7706,2
2,C9240,2
3,C5390,2
4,C5468,2
5,C6366,2
6,C3313,2
7,C2710,2
8,C5033,2
9,C6242,2


In [20]:
# ============================================
# Investigate Repeated Campaign IDs
# ============================================

repeated_ids = (
    df["Campaign_ID"]
    .value_counts()
)

repeated_ids = repeated_ids[
    repeated_ids > 1
].index

repeated_campaigns = (
    df[df["Campaign_ID"].isin(repeated_ids)]
    .sort_values(["Campaign_ID", "Date"])
)

display(
    repeated_campaigns[
        [
            "Campaign_ID",
            "Date",
            "Platform",
            "Content_Type",
            "Target_Age",
            "Target_Gender",
            "Region",
            "Budget",
            "Spend",
            "Conversions",
            "Revenue"
        ]
    ].head(30)
)

,Campaign_ID,Date,Platform,Content_Type,Target_Age,Target_Gender,Region,Budget,Spend,Conversions,Revenue
695,C1179,2024-11-19,Google,Text,18-24,Female,Asia,5180,5231.8,570,84360
744,C1179,2025-01-06,LinkedIn,Carousel,45-54,Male,Asia,8220,7891.2,1002,121242
439,C2099,2024-05-16,Instagram,Text,18-24,Male,Europe,3180,3084.6,486,91854
787,C2099,2024-12-22,Facebook,Text,45-54,Other,South America,9920,9920.0,598,99268
572,C2258,2024-02-10,Facebook,Text,18-24,Male,Africa,6770,6431.5,75,11625
889,C2258,2025-01-12,Google,Video,35-44,Male,Europe,9350,9069.5,1503,129258
982,C2710,2024-04-29,Google,Carousel,55+,Female,Africa,2290,2358.7,398,11940
395,C2710,2024-06-12,YouTube,Image,25-34,Other,Europe,5760,5875.2,933,34521
537,C2717,2024-04-15,YouTube,Image,45-54,Male,South America,8480,8056.0,935,105655
400,C2717,2024-08-20,Google,Carousel,35-44,Other,South America,4920,5018.4,50,6450


In [21]:
# ============================================
# Determine the actual row grain
# ============================================

print("Total rows:", len(df))

# Check duplicate Campaign_ID + Date combinations
campaign_date_counts = (
    df.groupby(["Campaign_ID", "Date"])
      .size()
      .reset_index(name="Row_Count")
)

print(
    "Duplicate Campaign_ID + Date combinations:",
    (campaign_date_counts["Row_Count"] > 1).sum()
)

# Check whether Campaign_ID + Date uniquely identifies a row
print(
    "Unique Campaign_ID + Date combinations:",
    campaign_date_counts.shape[0]
)

# Check complete duplicate rows
print(
    "Complete duplicate rows:",
    df.duplicated().sum()
)

# Show any duplicate Campaign_ID + Date combinations
display(
    campaign_date_counts[
        campaign_date_counts["Row_Count"] > 1
    ].head(20)
)

Total rows: 1000
Duplicate Campaign_ID + Date combinations: 0
Unique Campaign_ID + Date combinations: 1000
Complete duplicate rows: 0


,Campaign_ID,Date,Row_Count


In [27]:
# ============================================
# Create Final Cleaned Marketing Dataset
# ============================================

cleaned_df = df[
    [
        "Campaign_ID",
        "Budget",
        "Clicks",
        "Conversions",
        "Duration",
        "Platform",
        "Content_Type",
        "Target_Age",
        "Target_Gender",
        "Region",
        "Revenue",
        "Spend",
        "Date",
        "Impressions",
        "Calculated_CTR",
        "Calculated_CPC",
        "Calculated_Conversion_Rate",
        "Calculated_CPA",
        "Calculated_ROAS",
        "Budget_Utilization",
        "Budget_Overrun",
        "Budget_Status"
    ]
].copy()

output_path = (
    r"C:\Users\Lenovo\Desktop\Mahak Project"
    r"\Marketing Campaign Business Performance Analytics"
    r"\data\processed\Marketing_Campaign_Cleaned.csv"
)

cleaned_df.to_csv(output_path, index=False)

print("✅ Marketing_Campaign_Cleaned.csv created!")
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))

display(cleaned_df.head())

✅ Marketing_Campaign_Cleaned.csv created!
Rows: 1000
Columns: 22


,Campaign_ID,Budget,Clicks,Conversions,Duration,Platform,Content_Type,Target_Age,Target_Gender,Region,...,Date,Impressions,Calculated_CTR,Calculated_CPC,Calculated_Conversion_Rate,Calculated_CPA,Calculated_ROAS,Budget_Utilization,Budget_Overrun,Budget_Status
0,C3578,6390,401,174,20,Instagram,Carousel,35-44,Male,Europe,...,2025-01-19,8698,0.046103,16.094514,0.433915,37.091379,4.313671,1.01,63.9,Over Budget
1,C6702,9870,1286,821,28,LinkedIn,Text,55+,Male,Africa,...,2025-01-22,4496,0.286032,7.828460,0.638414,12.262363,12.721855,1.02,197.4,Over Budget
2,C9725,7700,1684,1060,15,Instagram,Video,35-44,Other,North America,...,2024-07-23,7935,0.212224,4.526722,0.629454,7.191509,25.446675,0.99,0.0,Within Budget
3,C9472,8420,444,308,25,Google,Text,25-34,Male,North America,...,2024-04-20,4620,0.096104,19.153604,0.693694,27.611039,2.824957,1.01,84.2,Over Budget
4,C7601,8470,1912,1428,9,Google,Text,25-34,Other,Europe,...,2024-08-07,5235,0.365234,4.208421,0.746862,5.634804,34.428882,0.95,0.0,Within Budget


In [28]:
import os

file_path = r"C:\Users\Lenovo\Desktop\Mahak Project\Marketing Campaign Business Performance Analytics\data\processed\Marketing_Campaign_Cleaned.csv"

print("File exists:", os.path.exists(file_path))

if os.path.exists(file_path):
    print("File size (MB):", round(os.path.getsize(file_path) / (1024 * 1024), 2))

File exists: True
File size (MB): 0.2


In [12]:
display(campaign_analysis)

,Campaign_ID,Platform,Content_Type,Target_Age,Target_Gender,Region,Spend,Conversions,Revenue,Calculated_CPA,Calculated_ROAS
0,C3578,Instagram,Carousel,35-44,Male,Europe,6453.9,174,27840,37.091379,4.313671
1,C6702,LinkedIn,Text,55+,Male,Africa,10067.4,821,128076,12.262363,12.721855
2,C9725,Instagram,Video,35-44,Other,North America,7623.0,1060,193980,7.191509,25.446675
3,C9472,Google,Text,25-34,Male,North America,8504.2,308,24024,27.611039,2.824957
4,C7601,Google,Text,25-34,Other,Europe,8046.5,1428,277032,5.634804,34.428882
...,...,...,...,...,...,...,...,...,...,...,...
995,C4071,Facebook,Carousel,18-24,Other,Europe,7696.0,136,24072,56.588235,3.127859
996,C2402,LinkedIn,Video,18-24,Other,Asia,3523.2,1258,206312,2.800636,58.558129
997,C3257,Google,Video,55+,Other,North America,5061.0,83,5146,60.975904,1.016795
998,C4315,Instagram,Image,55+,Other,Europe,8956.8,239,35372,37.476151,3.949178


In [13]:
top_10_roas = (
    campaign_analysis
    .sort_values("Calculated_ROAS", ascending=False)
    .head(10)
)

display(
    top_10_roas[
        ["Campaign_ID", "Platform", "Content_Type", "Calculated_ROAS"]
    ]
)

,Campaign_ID,Platform,Content_Type,Calculated_ROAS
831,C3181,YouTube,Image,93.901579
259,C1783,Facebook,Image,87.979251
875,C6062,Facebook,Video,83.306333
30,C2608,LinkedIn,Image,80.360036
92,C5887,Instagram,Video,73.668482
14,C2514,Facebook,Video,69.860702
178,C8825,LinkedIn,Text,66.993372
500,C5539,Google,Image,66.699029
837,C9923,YouTube,Carousel,62.971930
212,C5999,LinkedIn,Carousel,62.489462
